# Train a multi-task transformer on div + add + max + parity (5 seeds, P=149, WD=0.3)

Reproduces the training runs that underlie all figures in Part 2 of *Inside the grokked manifold*. Each seed trains one 4.8M-parameter transformer with a shared head on all four tasks simultaneously until either every task has grokked (≥ 95% test accuracy) plus a margin of 500 steps, or 10000 steps total.

**Wall time**: ~10–20 minutes per seed on a single A100 / T4. Five seeds run sequentially.

**Disk**: snapshots are written every `SNAP_EVERY` steps. With 5 seeds × ~50 snapshots × 19MB ≈ 5 GB on Colab disk if `SNAP_EVERY=200`. Set `SAVE_SNAPSHOTS = False` if you only want the final checkpoints.

Outputs go to `./out/seed_{N}/final.pt`, `./out/seed_{N}/snapshots/step_*.pt`, and `./out/summary.json`.

In [ ]:
# Cell 1 — config + setup
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, json, time, gc
from pathlib import Path
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

# === knobs ===
SEEDS = [1000, 1001, 1002, 1003, 1004]   # set [1000] for a quick single-seed check
SAVE_SNAPSHOTS = True                    # set False to keep only final.pt per seed
SNAP_EVERY = 200
MAX_STEPS = 10000
METRICS_EVERY = 10
GROK_THRESH = 0.95
POST_GROK_MARGIN = 500
BATCH = 512
PER_TASK_BATCH = BATCH // 4
WD = 0.3
LR = 1e-3

BASE_OUT = Path('./out'); BASE_OUT.mkdir(exist_ok=True)

P = 149
OP_DIV, OP_ADD, OP_MAX, OP_PARITY, EQ = P, P+1, P+2, P+3, P+4
VOCAB = P + 5
TASKS = ['div', 'add', 'max', 'parity']
TASK_OP = {'div': OP_DIV, 'add': OP_ADD, 'max': OP_MAX, 'parity': OP_PARITY}
TASK_COLORS = {'div': '#D45A2A', 'add': '#3A6E8C', 'max': '#5E9C5C', 'parity': '#9C5EBE'}
BASELINE_WD1 = {'max': 250, 'div': 3430, 'parity': 15230, 'add': 22780}

BG, FG, GRID, MUTED = '#FAFAF7', '#2A2A2A', '#E5E3DD', '#6E6E6E'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'savefig.facecolor': BG,
    'axes.edgecolor': FG, 'axes.linewidth': 0.5, 'axes.labelcolor': FG,
    'axes.titlecolor': FG, 'xtick.color': FG, 'ytick.color': FG, 'text.color': FG,
    'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.5, 'grid.alpha': 0.6,
})

In [ ]:
# Cell 2 — architecture + data (50/50 split per task)
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-8):
        super().__init__(); self.scale = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.scale

class GrokBlock(nn.Module):
    def __init__(self, d, nh, dropout=0.2):
        super().__init__()
        self.norm1 = RMSNorm(d); self.attn = nn.MultiheadAttention(d, nh, dropout=dropout, batch_first=True)
        self.norm2 = RMSNorm(d); self.w1 = nn.Linear(d, 4*d, bias=False)
        self.w2 = nn.Linear(4*d, d, bias=False); self.w3 = nn.Linear(d, 4*d, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        h = self.norm1(x); o, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.dropout(o)
        h2 = self.norm2(x); gate = F.silu(self.w1(h2))
        return x + self.dropout(self.w2(gate * self.w3(h2)))

class GrokModelShared(nn.Module):
    def __init__(self, vocab=VOCAB, d=384, nh=12, n_layers=2, out_p=P):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, d); self.pos_emb = nn.Embedding(4, d)
        self.blocks = nn.ModuleList([GrokBlock(d, nh) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)
        self.head = nn.Linear(d, out_p, bias=False)
    def forward(self, a, op_token, b):
        B = a.size(0)
        op = torch.full((B,), op_token, device=a.device, dtype=torch.long)
        eq = torch.full((B,), EQ, device=a.device, dtype=torch.long)
        tok = torch.stack([a, op, b, eq], dim=1)
        pos = torch.arange(4, device=a.device).unsqueeze(0).expand(B, -1)
        x = self.tok_emb(tok) + self.pos_emb(pos)
        for bl in self.blocks: x = bl(x)
        return self.head(self.norm_f(x)[:, -1, :])

def make_task_data(task, split_seed=0):
    aa, bb, cc = [], [], []
    for a in range(P):
        for b in range(P):
            if task == 'div' and b == 0: continue
            if task == 'div':    c = (a * pow(b, P-2, P)) % P
            elif task == 'add':  c = (a + b) % P
            elif task == 'max':  c = max(a, b)
            elif task == 'parity': c = (a + b) % 2
            aa.append(a); bb.append(b); cc.append(c)
    aa = torch.tensor(aa, device=device); bb = torch.tensor(bb, device=device); cc = torch.tensor(cc, device=device)
    g = torch.Generator(device=device); g.manual_seed(split_seed)
    perm = torch.randperm(len(aa), device=device, generator=g)
    nt = len(aa) // 2
    return (aa[perm[:nt]], bb[perm[:nt]], cc[perm[:nt]]), (aa[perm[nt:]], bb[perm[nt:]], cc[perm[nt:]])

data = {t: make_task_data(t) for t in TASKS}

@torch.no_grad()
def eval_all(m):
    m.eval(); accs = {}
    for t in TASKS:
        te = data[t][1]
        logits = m(te[0], TASK_OP[t], te[1])
        accs[t] = (logits.argmax(-1) == te[2]).float().mean().item()
    return accs

In [ ]:
# Cell 3 — training loop (resumes if final.pt already exists)
def train_seed(seed, out_dir):
    print(f'\n{"="*60}\n=== seed {seed} ===\n{"="*60}')
    final_path = out_dir / 'final.pt'
    if final_path.exists():
        ckpt = torch.load(final_path, map_location='cpu', weights_only=False)
        print(f'  loaded existing ckpt; grok={ckpt["grok_step"]}')
        return ckpt

    out_dir.mkdir(parents=True, exist_ok=True)
    if SAVE_SNAPSHOTS: (out_dir / 'snapshots').mkdir(exist_ok=True)

    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    m = GrokModelShared(d=384).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD, betas=(0.9, 0.98))
    pg = torch.Generator(device=device); pg.manual_seed(seed + 7)

    metrics = []
    grok_step = {t: None for t in TASKS}
    snap_steps = []
    t0 = time.time(); last_log = t0
    for step in range(1, MAX_STEPS + 1):
        m.train()
        if step <= 10:
            for grp in opt.param_groups: grp['lr'] = LR * step / 10
        total_loss = 0.0
        for t in TASKS:
            tr = data[t][0]
            pm = torch.randperm(len(tr[0]), device=device, generator=pg)[:PER_TASK_BATCH]
            logits = m(tr[0][pm], TASK_OP[t], tr[1][pm])
            total_loss = total_loss + F.cross_entropy(logits, tr[2][pm])
        opt.zero_grad(); total_loss.backward(); opt.step()

        if step % METRICS_EVERY == 0 or step == 1:
            test_accs = eval_all(m)
            metrics.append({'step': step, **{f'test_{t}': test_accs[t] for t in TASKS}})
            for t in TASKS:
                if grok_step[t] is None and test_accs[t] > GROK_THRESH:
                    grok_step[t] = step
                    print(f'  step {step}: {t.upper()} grokked')
        if SAVE_SNAPSHOTS and (step % SNAP_EVERY == 0 or step == 1):
            torch.save({n: p.data.cpu().clone() for n, p in m.named_parameters()},
                       out_dir / 'snapshots' / f'step_{step:06d}.pt')
            snap_steps.append(step)
        if time.time() - last_log > 30:
            acc_str = ', '.join(f'{t}={test_accs[t]:.3f}' for t in TASKS)
            print(f'  step {step:5d}: {acc_str} | grok={grok_step} ({time.time()-t0:.0f}s)')
            last_log = time.time()
        if all(grok_step[t] is not None for t in TASKS):
            stop_at = max(grok_step.values()) + POST_GROK_MARGIN
            if step >= stop_at: print(f'  step {step}: all 4 grokked + margin; stop.'); break

    final_accs = eval_all(m)
    ckpt = {
        'state_dict': {n: p.data.cpu().clone() for n, p in m.named_parameters()},
        'metrics': metrics, 'grok_step': grok_step, 'final_accs': final_accs,
        'config': {'P': P, 'wd': WD, 'lr': LR, 'TASKS': TASKS, 'seed': seed},
        'snap_steps': snap_steps,
    }
    torch.save(ckpt, final_path)
    print(f'  seed {seed} done; grok={grok_step}; final={final_accs}; wall {time.time()-t0:.0f}s')
    del m, opt; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return ckpt

all_ckpts = {}
for seed in SEEDS:
    out_dir = BASE_OUT / f'seed_{seed}'
    all_ckpts[seed] = train_seed(seed, out_dir)
print('\n========== ALL SEEDS DONE ==========')

In [ ]:
# Cell 4 — cross-seed stats + 4-panel training curves
grok_per_seed = {seed: all_ckpts[seed]['grok_step'] for seed in SEEDS}
final_per_seed = {seed: all_ckpts[seed]['final_accs'] for seed in SEEDS}

stats = {}
print(f'\n=== Grok timings across {len(SEEDS)} seed(s) @ WD={WD} ===')
print(f'{"task":>7} | mean ± std (range)    | baseline WD=1.0 | mean speedup')
for t in TASKS:
    vals = [grok_per_seed[s][t] for s in SEEDS if grok_per_seed[s][t] is not None]
    if vals:
        m_v, s_v = float(np.mean(vals)), float(np.std(vals))
        stats[t] = {'mean': m_v, 'std': s_v, 'min': min(vals), 'max': max(vals),
                    'speedup_vs_wd1': BASELINE_WD1[t] / m_v}
        print(f'{t:>7} | {m_v:.0f} ± {s_v:.0f} ({min(vals)}-{max(vals)})  | {BASELINE_WD1[t]:>10}     | {stats[t]["speedup_vs_wd1"]:.2f}x')
    else:
        stats[t] = None; print(f'{t:>7} | (none grokked)')

fig, axes = plt.subplots(1, 4, figsize=(15, 4), dpi=120, sharey=True)
for ax, t in zip(axes, TASKS):
    for seed in SEEDS:
        ms = all_ckpts[seed]['metrics']
        steps = [r['step'] for r in ms]
        acc = [r[f'test_{t}'] for r in ms]
        ax.plot(steps, acc, color=TASK_COLORS[t], linewidth=0.7, alpha=0.4)
    if stats[t] and stats[t]['mean']:
        ax.axvline(stats[t]['mean'], color=TASK_COLORS[t], linestyle='--', linewidth=1.5,
                   label=f'mean grok = {stats[t]["mean"]:.0f}')
    ax.axvline(BASELINE_WD1[t], color='gray', linestyle=':', linewidth=0.8,
               label=f'baseline WD=1 @ {BASELINE_WD1[t]}')
    ax.axhline(0.95, color=MUTED, linestyle=':', linewidth=0.4)
    ax.set_title(t, fontsize=11); ax.set_xlabel('step'); ax.set_ylim(-0.02, 1.05)
    ax.legend(fontsize=7, frameon=False, loc='lower right')
axes[0].set_ylabel('test accuracy')
plt.suptitle(f'Cross-seed training curves @ WD={WD} (P={P}, 4 tasks)', fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

with open(BASE_OUT / 'summary.json', 'w') as f:
    json.dump({
        'seeds': SEEDS,
        'baseline_wd1': BASELINE_WD1,
        'grok_per_seed': grok_per_seed,
        'final_per_seed': final_per_seed,
        'stats': stats,
        'config': {'wd': WD, 'lr': LR, 'p': P, 'tasks': TASKS},
    }, f, indent=2)
print(f'\nSaved: ./out/summary.json')

## What this notebook produces

Per-seed output structure:
```
out/
├── summary.json            cross-seed grok timings + final accuracies + speedups
└── seed_{1000..1004}/
    ├── final.pt            state dict + metrics + grok steps + snap step list
    └── snapshots/          (if SAVE_SNAPSHOTS) state dict every SNAP_EVERY steps
```

These are the inputs to the analysis notebooks: `walk_and_dictation.ipynb` (uses final.pt), `multilocation_analysis.ipynb` (uses snapshots), and the `index.html` data preprocessing.